In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

In [2]:
df = pd.read_parquet('CWRU_raw_data_0_load.parquet')

In [3]:
SEGMENT_LEN = 1024   # samples per window (~typical for CWRU)
STEP = 512 

In [4]:
segments, labels = [], []

for fault_name, group in df.groupby('fault'):
    signal = group['DE_data'].values
    for start in range(0, len(signal) - SEGMENT_LEN, STEP):
        seg = signal[start : start + SEGMENT_LEN]
        segments.append(seg)
        labels.append(fault_name)

In [5]:
X = np.array(segments, dtype=np.float32)   # shape: (N, 1024)
y_raw = np.array(labels)

# Normalize each segment independently (zero-mean, unit-std)
X = (X - X.mean(axis=1, keepdims=True)) / (X.std(axis=1, keepdims=True) + 1e-8)

In [6]:
# Encode labels  →  e.g. "B007", "IR014", "OR021", "Normal" etc.
le = LabelEncoder()
y = le.fit_transform(y_raw)               # integers 0..N_classes-1
num_classes = len(le.classes_)
print("Classes:", le.classes_)            # verify your 16 fault labels

# Reshape for Keras Conv1D  →  (N, timesteps, channels)
X = X[..., np.newaxis]                    # (N, 1024, 1)

# Split  →  60 / 20 / 20
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4,  stratify=y, random_state=42)
X_val,   X_test, y_val,  y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=42)
print(f"Train: {len(X_train)}  Val: {len(X_val)}  Test: {len(X_test)}")

Classes: ['07 B' '07 IR' '07 OR1' '07 OR2' '07 OR3' '14 B' '14 IR' '14 OR' '21 B'
 '21 IR' '21 OR1' '21 OR2' '21 OR3' '28 B' '28 IR' 'Normal']
Train: 2411  Val: 804  Test: 804


In [7]:
import tensorflow as tf
from tensorflow.keras import layers, models

def build_1d_cnn(input_shape, num_classes):
    inputs = tf.keras.Input(shape=input_shape)   # (1024, 1)
    x = inputs

    # Block 1 – learn low-level waveform features
    x = layers.Conv1D(32, kernel_size=64, strides=2, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 2
    x = layers.Conv1D(64, kernel_size=32, strides=1, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 3
    x = layers.Conv1D(128, kernel_size=16, strides=1, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(pool_size=2)(x)

    # Block 4 – high-level patterns
    x = layers.Conv1D(256, kernel_size=8, strides=1, padding='same', activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)       # replaces Flatten, reduces overfitting

    # Classifier head
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

model = build_1d_cnn((1024, 1), num_classes)
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1024, 1)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 512, 32)        │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512, 32)        │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 256, 32)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 256, 64)        │        65,600 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256, 64)        │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 128, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_2 (Conv1D)               │ (None, 128, 128)       │       131,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 128, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 64, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_3 (Conv1D)               │ (None, 64, 256)        │       262,400 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 64, 256)        │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 16)             │         2,064 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 498,160 (1.90 MB)

 Trainable params: 497,200 (1.90 MB)

 Non-trainable params: 960 (3.75 KB)

In [8]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

callbacks = [
    EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6),
    ModelCheckpoint('best_model.keras', save_best_only=True, monitor='val_accuracy')
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=callbacks
)

Epoch 1/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 4s 68ms/step - accuracy: 0.8586 - loss: 0.4843 - val_accuracy: 0.1754 - val_loss: 2.7765 - learning_rate: 0.0010
Epoch 2/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.9909 - loss: 0.0431 - val_accuracy: 0.1157 - val_loss: 5.3013 - learning_rate: 0.0010
Epoch 3/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 45ms/step - accuracy: 0.9959 - loss: 0.0171 - val_accuracy: 0.1169 - val_loss: 6.0440 - learning_rate: 0.0010
Epoch 4/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 53ms/step - accuracy: 0.9979 - loss: 0.0098 - val_accuracy: 0.1816 - val_loss: 5.8890 - learning_rate: 0.0010
Epoch 5/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 3s 82ms/step - accuracy: 0.9950 - loss: 0.0135 - val_accuracy: 0.2313 - val_loss: 5.9673 - learning_rate: 0.0010
Epoch 6/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 49ms/step - accuracy: 0.9971 - loss: 0.0153 - val_accuracy: 0.2823 - val_loss: 4.7877 - learning_rate: 0.0010
Epoch 7/100
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 48ms/step - accuracy: 0.9979 - loss: 0.0121 - 